# Análisis de la Copa Mundial de Fútbol 2022

Este notebook responde las preguntas del taller utilizando exclusivamente los archivos Parquet generados por `extractor_api.py`.

In [1]:
import pandas as pd

equipos = pd.read_parquet("equipos.parquet")
partidos = pd.read_parquet("partidos.parquet")
clasificacion = pd.read_parquet("clasificacion.parquet")

print("Equipos:", len(equipos))
print("Partidos:", len(partidos))
print("Registros de clasificación:", len(clasificacion))

Equipos: 32
Partidos: 64
Registros de clasificación: 32


## 1. ¿Cuántos equipos participaron en la competición?

In [2]:
cantidad_equipos = equipos["equipo_id"].nunique()
print("Cantidad de equipos participantes:", cantidad_equipos)

Cantidad de equipos participantes: 32


## 2. ¿Cuántos partidos se jugaron en cada ronda?

In [3]:
partidos_por_ronda = (
    partidos.groupby("ronda")
    .size()
    .reset_index(name="cantidad_partidos")
)
partidos_por_ronda

,ronda,cantidad_partidos
0,3rd Place Final,1
1,Final,1
2,Group Stage - 1,16
3,Group Stage - 2,16
4,Group Stage - 3,16
5,Quarter-finals,4
6,Round of 16,8
7,Semi-finals,2


## 3. ¿Cuál fue el partido con mayor cantidad total de goles?

In [4]:
partidos_con_goles = partidos.copy()
partidos_con_goles["total_goles"] = (
    partidos_con_goles["goles_local"].fillna(0)
    + partidos_con_goles["goles_visitante"].fillna(0)
)
maximo_goles = partidos_con_goles["total_goles"].max()
partidos_con_goles[partidos_con_goles["total_goles"] == maximo_goles][[
    "partido_id", "equipo_local_nombre", "equipo_visitante_nombre",
    "goles_local", "goles_visitante", "total_goles"
]]

,partido_id,equipo_local_nombre,equipo_visitante_nombre,goles_local,goles_visitante,total_goles
1,855735,England,Iran,6,2,8


## 4. ¿Cuál fue el equipo que anotó más goles?

In [5]:
goles_local = partidos[["equipo_local_id", "equipo_local_nombre", "goles_local"]].copy()
goles_local.columns = ["equipo_id", "nombre_equipo", "goles"]

goles_visitante = partidos[["equipo_visitante_id", "equipo_visitante_nombre", "goles_visitante"]].copy()
goles_visitante.columns = ["equipo_id", "nombre_equipo", "goles"]

goles_por_equipo = (
    pd.concat([goles_local, goles_visitante])
    .groupby(["equipo_id", "nombre_equipo"], as_index=False)["goles"]
    .sum()
)
maximo = goles_por_equipo["goles"].max()
goles_por_equipo[goles_por_equipo["goles"] == maximo]

,equipo_id,nombre_equipo,goles
1,2,France,16


## 5. ¿Cuál fue el equipo que ganó más partidos?

In [6]:
ganadores_locales = partidos[partidos["gano_local"] == True][[
    "equipo_local_id", "equipo_local_nombre"
]].copy()
ganadores_locales.columns = ["equipo_id", "nombre_equipo"]

ganadores_visitantes = partidos[partidos["gano_visitante"] == True][[
    "equipo_visitante_id", "equipo_visitante_nombre"
]].copy()
ganadores_visitantes.columns = ["equipo_id", "nombre_equipo"]

victorias = (
    pd.concat([ganadores_locales, ganadores_visitantes])
    .groupby(["equipo_id", "nombre_equipo"])
    .size()
    .reset_index(name="cantidad_victorias")
)
maximo = victorias["cantidad_victorias"].max()
victorias[victorias["cantidad_victorias"] == maximo]

,equipo_id,nombre_equipo,cantidad_victorias
17,26,Argentina,6


## 6. ¿Cuál fue el equipo con mejor diferencia de gol en cada grupo?

In [7]:
mejor_diferencia = clasificacion.groupby("grupo")["diferencia_gol"].transform("max")
clasificacion[clasificacion["diferencia_gol"] == mejor_diferencia][[
    "grupo", "equipo_id", "nombre_equipo", "diferencia_gol"
]]

,grupo,equipo_id,nombre_equipo,diferencia_gol
0,Group A,1118,Netherlands,4
4,Group B,10,England,7
8,Group C,26,Argentina,3
12,Group D,2,France,3
17,Group E,9,Spain,6
20,Group F,31,Morocco,3
21,Group F,3,Croatia,3
24,Group G,6,Brazil,2
28,Group H,27,Portugal,2


## 7. ¿Qué equipos terminaron en la primera posición de cada grupo?

In [8]:
clasificacion[clasificacion["posicion"] == 1][[
    "grupo", "equipo_id", "nombre_equipo", "puntos"
]]

,grupo,equipo_id,nombre_equipo,puntos
0,Group A,1118,Netherlands,7
4,Group B,10,England,7
8,Group C,26,Argentina,6
12,Group D,2,France,6
16,Group E,12,Japan,6
20,Group F,31,Morocco,7
24,Group G,6,Brazil,6
28,Group H,27,Portugal,6


## 8. ¿Cuál fue el estadio en el que se disputaron más partidos?

In [9]:
partidos_por_estadio = (
    partidos.groupby(["estadio_id", "estadio_nombre"])
    .size()
    .reset_index(name="cantidad_partidos")
)
maximo = partidos_por_estadio["cantidad_partidos"].max()
partidos_por_estadio[partidos_por_estadio["cantidad_partidos"] == maximo]

,estadio_id,estadio_nombre,cantidad_partidos
0,22430,Khalifa International Stadium,8


## 9. ¿Existen partidos duplicados según partido_id?

In [10]:
duplicados = partidos[partidos.duplicated(subset="partido_id", keep=False)]
print("¿Existen partidos duplicados?:", not duplicados.empty)
duplicados

¿Existen partidos duplicados?: False


,partido_id,competencia_id,competencia_nombre,temporada,ronda,fecha_partido,zona_horaria,estado_partido,minuto_transcurrido,arbitro,...,equipo_visitante_id,equipo_visitante_nombre,gano_local,gano_visitante,goles_local,goles_visitante,penales_local,penales_visitante,fecha_extraccion,endpoint_origen


## 10. ¿Cuántos valores nulos tiene cada columna de cada archivo?

In [11]:
nulos = pd.concat(
    [
        equipos.isnull().sum().rename("equipos"),
        partidos.isnull().sum().rename("partidos"),
        clasificacion.isnull().sum().rename("clasificacion"),
    ],
    axis=1,
).fillna(0).astype(int)
nulos

,equipos,partidos,clasificacion
equipo_id,0,0,0
nombre_equipo,0,0,0
codigo_equipo,0,0,0
pais,0,0,0
anio_fundacion,0,0,0
es_seleccion_nacional,0,0,0
logo_url,0,0,0
competencia_id,0,0,0
temporada,0,0,0
fecha_extraccion,0,0,0
